In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from utils.model_saver import *
from utils.model_classes import KNNModel

PROJECT_ROOT = Path().resolve().parent.parent
print(f"Project root: {PROJECT_ROOT}")

INITIAL_FEATURES_PATH = PROJECT_ROOT / 'data' / 'normal_features'
INITIAL_TRAIN_PATH = INITIAL_FEATURES_PATH / 'train.parquet'
INITIAL_VAL_PATH = INITIAL_FEATURES_PATH / 'val.parquet'
INITIAL_TEST_PATH = INITIAL_FEATURES_PATH / 'test.parquet'

GRAPH_PATH = PROJECT_ROOT / 'data' / 'graph_features'
GRAPH_TRAIN_PATH = GRAPH_PATH / 'final_train.parquet'
GRAPH_VAL_PATH = GRAPH_PATH / 'final_val.parquet'
GRAPH_TEST_PATH = GRAPH_PATH / 'final_test.parquet'

TEXTUAL_PATH = PROJECT_ROOT / 'data' / 'textual_features'
text_emb_64 = TEXTUAL_PATH / 'textual_embeddings_64.parquet'

OUTPUT_DIR =  PROJECT_ROOT / 'data' / 'combined_features'

RANDOM_STATE = 42

import torch
# Detect device: 'cuda' if available, else 'cpu'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# for parallelization
N_JOBS = -1

Using device: cuda
Project root: /home/tommaso/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/M-P6203E-DataProjects-Hackaton3_P1
Using device: cuda


# Combine features: graph + textual + initial features

In [2]:
ID_COLUMNS = ["article_id", "ref_id"]
USELESS_COLUMNS = ["split"]

In [3]:
def remove_prefix_cols(df, prefix='keywords_'):
    """ 
    Remove all columns where the name starts with the specified prefix.
    """
    # Use list comprehension to identify columns to drop
    cols_to_drop = [c for c in df.columns if c.startswith(prefix)]
    
    return df.drop(columns=cols_to_drop, inplace=False)

## 1. Train data

### Import

In [4]:
textual_df = pd.read_parquet(text_emb_64)
split_series = textual_df["split"].astype(str).str.lower()
textual_df_train = textual_df[split_series == "train"].copy()

graph_train = pd.read_parquet(GRAPH_TRAIN_PATH)
initial_features_train = pd.read_parquet(INITIAL_TRAIN_PATH)

### Remove columns useless or redundant
- Since keywords are just included in the transformer the ones in the initial features hashed are redundant, then they should be removed.
- remove split column


In [5]:
initial_features_train = remove_prefix_cols(initial_features_train, prefix='keywords_')
initial_features_train.drop(columns=USELESS_COLUMNS, errors="ignore")

,article_id,ref_id,year_article,n_citation_article,year_ref,is_reference_valid,n_keywords_article,n_keywords_ref,n_authors_article,n_authors_ref,...,lang_ref_Other,lang_ref_de,lang_ref_en,lang_ref_es,lang_ref_fr,lang_ref_pt,doc_type_article_conference,doc_type_article_journal,doc_type_ref_conference,doc_type_ref_journal
0,53e99f0ab7602d97027d6a89,53e99ff0b7602d97028d14d3,1978.0,12.0,1977.0,1,2,0,1,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,53e9bd81b7602d9704a24d06,557f4d4f6fee0fe990cb035f,1996.0,5.0,1993.0,1,23,11,2,4,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,539087fe20f70186a0d75db6,539087ae20f70186a0d4cf5a,2000.0,21.0,1992.0,1,5,5,3,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,539087fe20f70186a0d75db6,5390878e20f70186a0d3a260,2000.0,21.0,1989.0,1,5,12,3,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,539087fe20f70186a0d75db6,539087cb20f70186a0d58fe1,2000.0,21.0,1990.0,1,5,10,3,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2162515,53e9bde2b7602d9704a91def,53908aac20f70186a0da7f23,2002.0,21.0,1995.0,0,5,3,2,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2162516,53e9b5b6b7602d97040fba6c,5390878320f70186a0d33894,2003.0,9.0,1988.0,0,24,1,3,1,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2162517,53e9b5b6b7602d97040fba6c,53908b1820f70186a0db4f04,2003.0,9.0,2000.0,0,24,4,3,4,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2162518,53e9a146b7602d9702a39e8d,53e9a539b7602d9702e59e18,2002.0,27.0,2001.0,0,7,7,3,3,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


### Merge all

In [6]:
df_train = initial_features_train.merge(graph_train, on=["article_id", "ref_id", "is_reference_valid"], how="left")
df_train = df_train.merge(textual_df_train, on=["article_id", "ref_id", "is_reference_valid"], how="left")

In [7]:
del graph_train, initial_features_train, textual_df_train

## 2. Validation data

### Import

In [8]:
textual_df_val = textual_df[split_series == "validation"].copy()
graph_val = pd.read_parquet(GRAPH_VAL_PATH)
initial_features_val = pd.read_parquet(INITIAL_VAL_PATH)

### Remove columns useless or redundant
- Since keywords are just included in the transformer the ones in the initial features hashed are redundant, then they should be removed.
- remove split column


In [9]:
initial_features_val = remove_prefix_cols(initial_features_val, prefix='keywords_')
initial_features_val.drop(columns=USELESS_COLUMNS, errors="ignore")

,article_id,ref_id,year_article,n_citation_article,year_ref,is_reference_valid,n_keywords_article,n_keywords_ref,n_authors_article,n_authors_ref,...,lang_ref_Other,lang_ref_de,lang_ref_en,lang_ref_es,lang_ref_fr,lang_ref_pt,doc_type_article_conference,doc_type_article_journal,doc_type_ref_conference,doc_type_ref_journal
0,56d8c6addabfae2eee533da3,56d87f43dabfae2eee59ffc6,2012.0,3.0,2011.0,1,5,5,2,4,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,53e9b532b7602d9704068778,53909eef20f70186a0e361ad,2011.0,20.0,2007.0,1,5,8,2,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,53e9b532b7602d9704068778,573695696e3b12023e48edff,2011.0,20.0,2007.0,1,5,5,2,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,53e9b532b7602d9704068778,539098b820f70186a0e0b338,2011.0,20.0,2005.0,1,5,10,2,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,53e9b532b7602d9704068778,5390ae2e20f70186a0ec8984,2011.0,20.0,2010.0,1,5,5,2,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391237,53e9a86fb7602d97031bb869,53e9bc88b7602d970490a715,2010.0,29.0,2010.0,0,0,5,3,5,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
391238,53e9a86fb7602d97031bb869,573696266e3b12023e5371a9,2010.0,29.0,2012.0,0,0,0,3,4,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
391239,5390be6620f70186a0f4d20a,53e9ab5ab7602d97034f033f,2013.0,11.0,2013.0,0,4,0,4,7,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
391240,55a6411e65ce054aad61d0d8,5390adfc20f70186a0ec5017,2014.0,129.0,2010.0,0,4,7,3,8,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


### Merge all

In [10]:
df_val = initial_features_val.merge(graph_val, on=["article_id", "ref_id", "is_reference_valid"], how="left")
df_val = df_val.merge(textual_df_val, on=["article_id", "ref_id", "is_reference_valid"], how="left")

In [11]:
del graph_val, initial_features_val, textual_df_val

## 3. Test data

### Import

In [12]:
textual_df_test = textual_df[split_series == "test"].copy()

graph_test = pd.read_parquet(GRAPH_TEST_PATH)
initial_features_test = pd.read_parquet(INITIAL_TEST_PATH)

### Remove columns useless or redundant
- Since keywords are just included in the transformer the ones in the initial features hashed are redundant, then they should be removed.
- remove split column


In [13]:
initial_features_test = remove_prefix_cols(initial_features_test, prefix='keywords_')
initial_features_test.drop(columns=USELESS_COLUMNS, errors="ignore")

,article_id,ref_id,year_article,n_citation_article,year_ref,is_reference_valid,n_keywords_article,n_keywords_ref,n_authors_article,n_authors_ref,...,lang_ref_Other,lang_ref_de,lang_ref_en,lang_ref_es,lang_ref_fr,lang_ref_pt,doc_type_article_conference,doc_type_article_journal,doc_type_ref_conference,doc_type_ref_journal
0,6426ed4790e50fcafd445e7f,5d7f5db647c8f76646dc5cc6,2022.0,14.0,2019.0,1,3,4,5,9,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,5ff8836091e011c832672f54,57d063e0ac44367354294940,2020.0,12.0,2016.0,1,6,0,5,5,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,66ecda9201d2a3fbfcdf8856,5f35115a91e011e2b713e155,2024.0,35.0,2020.0,1,5,14,5,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,66ecda9201d2a3fbfcdf8856,6456389bd68f896efacf6bbb,2024.0,35.0,2023.0,1,5,4,5,6,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,61dd5fb55244ab9dcb8bddef,5a39cb9a0cf29b02debe17e8,2021.0,27.0,2018.0,1,5,6,4,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
396377,5e56432693d709897cd2fbcd,5e4671853a55ac121dd335a5,2019.0,22.0,2016.0,0,3,0,6,3,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
396378,6103d7c491e01159791b235f,65781f0d939a5f40824c131a,2021.0,8.0,2024.0,0,5,4,3,3,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
396379,66b4374a01d2a3fbfcc63b7b,5a73cb6a17c44a0b303590f3,2024.0,22.0,2017.0,0,0,3,25,4,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
396380,5df0be543a55ac84bd7f488f,6389c6ee90e50fcafdc9c00d,2022.0,27.0,2022.0,0,11,12,3,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


### Merge all

In [14]:
df_test = initial_features_test.merge(graph_test, on=["article_id", "ref_id", "is_reference_valid"], how="left")
df_test = df_test.merge(textual_df_test, on=["article_id", "ref_id", "is_reference_valid"], how="left")

In [15]:
del graph_test, initial_features_test, textual_df_test

## 4. Export data

In [16]:
print("\nSplit sizes train:", {df_train.shape})
print("\nSplit sizes validation:", {df_val.shape})
print("\nSplit sizes test:", {df_test.shape})

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_train.to_parquet(OUTPUT_DIR / "train.parquet", index=False)
df_val.to_parquet(OUTPUT_DIR / "val.parquet", index=False)
df_test.to_parquet(OUTPUT_DIR / "test.parquet", index=False)


Split sizes train: {(2162534, 173)}

Split sizes validation: {(391242, 173)}

Split sizes test: {(396386, 173)}
